In [ ]:
import pandas as pd
import json
import networkx as nx
from openai import OpenAI

# Initialize client - keeping API key as a variable for easy swapping
client = OpenAI(api_key="YOUR_OPENAI_API_KEY")

def extract_product_metadata(label):
    """
    Passes raw product strings through GPT to derive structured 
    semantic categories and attributes.
    """
    # Using a f-string block for the prompt to keep it clean
    sys_prompt = (
        "You are a retail data parser. Categorize the product provided into "
        "a JSON format with: 'category', 'sub_category', and 'tags'."
    )
    
    query = f"Product: {label}"
    
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": query}
        ],
        response_format={"type": "json_object"}
    )
    
    return json.loads(completion.choices[0].message.content)

def run_mapping_pipeline(input_csv):
    # Load dataset - assuming a standard CSV structure
    data = pd.read_csv(input_csv)
    
    # Initialize Directed Graph (DiGraph) for hierarchy modeling
    product_graph = nx.DiGraph()
    semantic_registry = []

    # Iterate through the dataframe (using a subset [:5] for testing)
    for name in data['product_name'].iloc[:10]:
        try:
            # Get semantic data
            meta = extract_product_metadata(name)
            
            # Build graph nodes and edges
            # Logic: Root -> Dept -> Class -> Product
            dept = meta.get('category')
            prod_class = meta.get('sub_category')
            
            product_graph.add_edge("Store_Front", dept)
            product_graph.add_edge(dept, prod_class)
            product_graph.add_edge(prod_class, name)
            
            semantic_registry.append({
                "raw_name": name,
                "metadata": meta
            })
            
            print(f"Processed: {name} -> {prod_class}")
            
        except Exception as e:
            print(f"Error processing {name}: {e}")

    # Save output to JSON for downstream use or Neo4j ingestion
    with open('retail_map_output.json', 'w') as out_file:
        json.dump(semantic_registry, out_file, indent=4)
        
    print("\nPipeline execution finished. Graph and JSON generated.")
    return product_graph

if __name__ == "__main__":
    # Replace with your actual CSV path from Kaggle
    results_graph = run_mapping_pipeline("retail_data.csv")